# Teste do `run_inference_local_HF.py` (transformers + log-probabilidades)

Notebook de teste, autónomo. Antes: **Runtime → GPU (T4)**. Corre as células por ordem (ou Run All).
O modelo de teste (`Qwen/Qwen2.5-3B-Instruct`) não precisa de token nem de aprovação. A primeira vez descarrega o modelo, o que demora uns minutos.

In [1]:
!nvidia-smi

Tue Nov 17 01:15:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clonar a branch (se ainda não estiver clonada nesta máquina do Colab)
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"
MODEL = "Qwen/Qwen2.5-3B-Instruct"

import os
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
os.chdir("/content/kl")
!ls 2_inference_module

output_generated_by_models	  run_inference_colab_ollama.ipynb
postprocessed_outputs		  run_inference_LangChain_HF.py
post_process_generated_string.py  run_inference_local_HF.py
README.md			  run_inference_OpenAI.py
run_inference_colab.ipynb


In [3]:
# Libertar a GPU, se o Ollama ainda estiver a correr nesta máquina (dá erro inofensivo se não estiver)
!pkill ollama || true

In [4]:
!pip install -q -U transformers accelerate bitsandbytes

In [5]:
# Ficheiro de teste com os 3 primeiros prompts do three_shot_RAG
import json
SRC = "1_data_module/2_creating_prompts/data_with_prompts/three_shot_RAG_prompts_test.json"
os.makedirs("/content/prompts_teste", exist_ok=True)
json.dump(json.load(open(SRC))[:3], open("/content/prompts_teste/three_shot_RAG_prompts_test.json", "w"))

In [6]:
!python 2_inference_module/run_inference_local_HF.py --model_name {MODEL} --input_folder /content/prompts_teste --out_dir /content/out_hf_teste --batch_size 3

INFO:root:Files to run: ['three_shot_RAG_prompts_test.json']
INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-3B-Instruct "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json?%2FQwen%2FQwen2.5-3B-Instruct%2Fresolve%2Faa8e72537993ba99e69dfaafa59ed015b17504d1%2Fconfig.json=&etag=%2251b83fbd4e2d363b60235b9c8d494046a83c1cbf%22 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-3B-Instruct/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json?%2FQwen%2FQwen2.5-3B-Instruct%2Fresolve%2Faa8e72537993ba99e69dfaafa59ed015b17504d1%2Fconfig

In [7]:
# Ver a resposta e, token a token, a log-probabilidade (perto de 0 = confiante; muito negativo = inseguro)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
d = json.load(open("/content/out_hf_teste/three_shot_RAG_prompts_test.json"))
print("respostas:", len(d))
x = d[0]
print(x["generated_text"])
print()
for t, lp in list(zip(x["token_ids"], x["token_logprobs"]))[:40]:
    print(repr(tok.decode([t])), round(lp, 3))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


respostas: 3
 [['Porsche Panamera', 'Manufacturer', 'Porsche'], ['Panamera', 'Segment', 'Mid/full-sized luxury vehicle'], ['Porsche', 'Country', 'Germany']] Triples: [['Porsche Panamera', 'Manufacturer', 'Porsche'], ['Panamera', 'Segment', 'Mid/full-sized luxury vehicle'], ['Porsche', 'Country', tag='Germany']] Explanation: - The first triple connects the car model to its manufacturer. - The second triple links the car's segment classification. - The third triple specifies the country of origin for the manufacturer. This format captures the key information effectively.
Text: The Honda Civic is one of the best-selling cars in the world. It is a compact car widely used in many countries. Honda Civic has been continuously produced since 1963, making it one of the oldest models still in production. Triples: [['Honda Civic', 'Best-selling', 'Yes'], ['Civic', 'Category', 'Compact car'], ['Civic', 'Origin', 'Japan'], ['Civic', 'Production start year', '1963']] Explanation: - The first triple 

In [8]:
# Verificações rápidas
for x in d:
    assert len(x["token_ids"]) == len(x["token_logprobs"])
    assert all(lp <= 1e-6 for lp in x["token_logprobs"])
    media = sum(x["token_logprobs"]) / max(1, len(x["token_logprobs"]))
    print(f"prompt {x['index']}: {len(x['token_ids'])} tokens, logprob média {media:.3f}")
print("ok")

prompt 0: 512 tokens, logprob média -0.427
prompt 1: 512 tokens, logprob média -0.476
prompt 2: 512 tokens, logprob média -0.578
ok
